<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **SpaceX Falcon 9 First Stage Landing Prediction**

## Lab 2: Data Wrangling

**Author:** Ahmad Waziri

In this notebook we perform exploratory data analysis to find patterns in the data and determine
the training label. Several outcomes correspond to a booster that did not land successfully
(e.g. `False Ocean`, `False RTLS`, `False ASDS`), and others correspond to a successful landing
(`True Ocean`, `True RTLS`, `True ASDS`). We convert these outcomes into a binary `Class` label:
`1` for a successful landing, `0` otherwise.

In [1]:
import pandas as pd
import numpy as np

### Data Analysis

Load the SpaceX dataset produced by the data-collection notebook.

In [1]:
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
df.head(10)

   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite      Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount Serial   Longitude   Latitude
0             1  2010-06-04       Falcon 9  6104.959412   LEO  CCAFS SLC 40    None None        1     False   False  False        NaN    1.0            0  B0003  -80.577366  28.561857
1             2  2012-05-22       Falcon 9   525.000000   LEO  CCAFS SLC 40    None None        1     False   False  False        NaN    1.0            0  B0005  -80.577366  28.561857
2             3  2013-03-01       Falcon 9   677.000000   ISS  CCAFS SLC 40    None None        1     False   False  False        NaN    1.0            0  B0007  -80.577366  28.561857
3             4  2013-09-29       Falcon 9   500.000000    PO   VAFB SLC 4E  False Ocean        1     False   False  False        NaN    1.0            0  B1003 -120.610829  34.632093
4             5  2013-12-03       Falcon 9  3170.000000   GTO  CCAFS SLC 40    N

Identify and calculate the percentage of missing values in each attribute.

In [1]:
df.isnull().sum()/len(df)*100

FlightNumber       0.000000
Date               0.000000
BoosterVersion     0.000000
PayloadMass        0.000000
Orbit              0.000000
LaunchSite         0.000000
Outcome            0.000000
Flights            0.000000
GridFins           0.000000
Reused             0.000000
Legs               0.000000
LandingPad        28.888889
Block              0.000000
ReusedCount        0.000000
Serial             0.000000
Longitude          0.000000
Latitude           0.000000

Identify which columns are numerical and which are categorical.

In [1]:
df.dtypes

FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block             float64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64

### TASK 1: Calculate the number of launches on each site

The data contains several SpaceX launch facilities: Cape Canaveral Space Launch Complex 40
(`CCAFS SLC 40`), Vandenberg Air Force Base Space Launch Complex 4E (`VAFB SLC 4E`), and Kennedy
Space Center Launch Complex 39A (`KSC LC 39A`). The location of each launch is stored in the
`LaunchSite` column.

We use `value_counts()` to determine the number of launches on each site.

In [1]:
df['LaunchSite'].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13

### TASK 2: Calculate the number and occurrence of each orbit

Each launch targets a dedicated orbit. We use `value_counts()` on the `Orbit` column to see how
many launches targeted each orbit type.

In [1]:
df['Orbit'].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1

### TASK 3: Calculate the number and occurrence of each mission outcome

We use `value_counts()` on the `Outcome` column, then inspect each unique outcome.

In [1]:
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1

In [1]:
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


`True Ocean` means the first stage successfully landed in a specific ocean region, while
`False Ocean` means it landed there unsuccessfully (i.e. it landed, but the attempt failed).
`True RTLS`/`False RTLS` refer to a ground-pad landing, and `True ASDS`/`False ASDS` refer to a
drone-ship landing. `None ASDS` and `None None` represent no landing attempt at all.

We build the set of outcomes where the booster did **not** land successfully.

In [1]:
bad_outcomes = set(landing_outcomes.keys()[[1, 3, 5, 6, 7]])
bad_outcomes

{'False RTLS', 'False Ocean', 'None None', 'None ASDS', 'False ASDS'}

### TASK 4: Create a landing-outcome label from the `Outcome` column

Using `Outcome`, build a list where the element is `0` if the corresponding row's outcome is in
`bad_outcomes`, and `1` otherwise. This becomes the `landing_class` training label.

In [1]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]

`Class = 0` means the first stage did not land successfully; `Class = 1` means it landed successfully.

In [1]:
df['Class'] = landing_class
df[['Class']].head(8)

   Class
0      0
1      0
2      0
3      0
4      0
5      0
6      1
7      1

In [1]:
df.head(5)

   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite      Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount Serial   Longitude   Latitude  Class
0             1  2010-06-04       Falcon 9  6104.959412   LEO  CCAFS SLC 40    None None        1     False   False  False        NaN    1.0            0  B0003  -80.577366  28.561857      0
1             2  2012-05-22       Falcon 9   525.000000   LEO  CCAFS SLC 40    None None        1     False   False  False        NaN    1.0            0  B0005  -80.577366  28.561857      0
2             3  2013-03-01       Falcon 9   677.000000   ISS  CCAFS SLC 40    None None        1     False   False  False        NaN    1.0            0  B0007  -80.577366  28.561857      0
3             4  2013-09-29       Falcon 9   500.000000    PO   VAFB SLC 4E  False Ocean        1     False   False  False        NaN    1.0            0  B1003 -120.610829  34.632093      0
4             5  2013-12-03       Falcon 9  3

Determine the overall success rate:

In [1]:
df["Class"].mean()

np.float64(0.6666666666666666)

## Conclusion

Across the 90 launch records, the first-stage landing succeeded on roughly two-thirds of attempts.
Cape Canaveral SLC-40 and Kennedy LC-39A hosted the majority of launches, and most missions targeted
low-Earth-orbit-family destinations (LEO, ISS, VLEO) or GTO. We export the labeled dataset for the
next stage of the pipeline (exploratory visualization and feature engineering).

In [1]:
df.to_csv("dataset_part_2.csv", index=False)
print("Saved dataset_part_2.csv with", df.shape[0], "rows and", df.shape[1], "columns")

Saved dataset_part_2.csv with 90 rows and 18 columns
